# VocalCoach Colab Training

Multi-task conformer training: pitch + VAD + technique (+ quality, note heads in later stages).

**Training strategy:** Differential LR throughout Stage 2+ — backbone gets a low LR to protect
pitch/VAD representations while technique/quality heads train faster. No frozen-backbone probe.

**Checkpoint strategy:** Checkpoints write to local disk (`/content/runs/`) for fast I/O.
A Drive-copy cell runs after each stage. If the session disconnects, re-run cells 1–4 and
`--resume` picks up from the last Drive checkpoint automatically.

**Run cells in order.** Cells 1–4 are setup (re-run at the start of every new session).

---

### Data upload (one-time, from local machine)

```bash
cd ~/NanoPitch-MusicalAI
zip -1 NanoPitch_data.zip \
    data/clean.npz \
    data/noise.npz \
    data/test.npz \
    data/vocalset/technique_train.npz \
    data/vocalset/technique_test.npz \
    data/annotated_vocalset/note_train.npz \
    data/annotated_vocalset/note_test.npz \
    data/gtsinger_technique/technique_gtsinger_train.npz \
    data/gtsinger_technique/technique_gtsinger_test.npz \
    data/gtsinger_technique/technique_train.npz
```

Upload `NanoPitch_data.zip` to `My Drive/musicalAI/vocalCoach/`.

Expected Drive layout after runs complete:
```
My Drive/musicalAI/vocalCoach/
  NanoPitch_data.zip
  NanoPitch-runs/
    stage1_conformer_96/          ← local 64-hidden Stage 1 backbone (best VDR=88.3%)
    stage1_conformer_128/         ← W1: wider backbone (in progress)
    stage2_w1_joint_difflr/       ← Stage 2 off best W1 backbone
    stage2_w1_difflr_v2/          ← continued difflr
    stage2_w1_supcon/             ← GTSinger SupCon
    stage2_w1_quality/            ← quality scoring head
```

## Cell 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH_SIZE = 64 if vram_gb > 30 else 32 if vram_gb > 15 else 16
NUM_WORKERS = 8
print(f"\nRecommended batch_size={BATCH_SIZE}, num_workers={NUM_WORKERS}")

## Cell 2 — Mount Drive and extract data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os, shutil

DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'
RUNS_DIR   = f'{DRIVE_ROOT}/NanoPitch-runs'
os.makedirs(RUNS_DIR, exist_ok=True)

# Expected destinations after extraction (zip retains data/subdir/ paths)
files_needed = {
    'data/clean.npz':                                      '/content/data/clean.npz',
    'data/noise.npz':                                      '/content/data/noise.npz',
    'data/test.npz':                                       '/content/data/test.npz',
    'data/vocalset/technique_train.npz':                   '/content/data/vocalset/technique_train.npz',
    'data/vocalset/technique_test.npz':                    '/content/data/vocalset/technique_test.npz',
    'data/annotated_vocalset/note_train.npz':              '/content/data/annotated_vocalset/note_train.npz',
    'data/annotated_vocalset/note_test.npz':               '/content/data/annotated_vocalset/note_test.npz',
    'data/gtsinger_technique/technique_gtsinger_train.npz':'/content/data/gtsinger_technique/technique_gtsinger_train.npz',
    'data/gtsinger_technique/technique_gtsinger_test.npz': '/content/data/gtsinger_technique/technique_gtsinger_test.npz',
    'data/gtsinger_technique/technique_train.npz':         '/content/data/gtsinger_technique/technique_train.npz',
}

missing = [name for name, dest in files_needed.items() if not os.path.exists(dest)]

if missing:
    print(f"Extracting {len(missing)} file(s) from NanoPitch_data.zip...")
    with zipfile.ZipFile(f'{DRIVE_ROOT}/NanoPitch_data.zip', 'r') as z:
        for name in missing:
            dest_dir = os.path.dirname(files_needed[name])
            os.makedirs(dest_dir, exist_ok=True)
            # Extract preserving path then move to /content/
            z.extract(name, '/content/')
            print(f"  {name} → {files_needed[name]}")
else:
    print("All data files already present — skipping extraction.")

!ls -lh /content/data/
!ls -lh /content/data/vocalset/
!ls -lh /content/data/annotated_vocalset/
!ls -lh /content/data/gtsinger_technique/

## Cell 3 — Clone or update repo and install dependencies

In [ ]:
import os

REPO_DIR = '/content/NanoPitch-MusicalAI'
REPO_URL = 'https://github.com/rajat17-personal/NanoPitch-MusicalAI'
BRANCH   = 'feat/finalProject'

if os.path.isdir(f'{REPO_DIR}/.git'):
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {BRANCH}
else:
    print("Cloning repo...")
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -r requirements.txt --quiet
print("Setup complete.")

## Cell 4 — Verify data loads correctly

In [ ]:
import numpy as np

def check(label, path, key='lengths'):
    d = np.load(path)
    print(f"{label:<45} clips={d[key].shape[0]}  keys={list(d.keys())}")

check('clean.npz',                          '/content/data/clean.npz')
check('noise.npz',                          '/content/data/noise.npz')
check('test.npz',                           '/content/data/test.npz', key='clips')
check('vocalset/technique_train.npz',       '/content/data/vocalset/technique_train.npz')
check('vocalset/technique_test.npz',        '/content/data/vocalset/technique_test.npz')
check('annotated_vocalset/note_train.npz',  '/content/data/annotated_vocalset/note_train.npz')
check('annotated_vocalset/note_test.npz',   '/content/data/annotated_vocalset/note_test.npz')
check('gtsinger_technique/train.npz',       '/content/data/gtsinger_technique/technique_train.npz')
check('gtsinger_technique/gtsinger_train',  '/content/data/gtsinger_technique/technique_gtsinger_train.npz')
check('gtsinger_technique/gtsinger_test',   '/content/data/gtsinger_technique/technique_gtsinger_test.npz')

---
## Helper: save run to Drive

Reusable function — call after any training cell completes.

```python
save_to_drive("run_name")
```

In [ ]:
import shutil, os

DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'
RUNS_DIR   = f'{DRIVE_ROOT}/NanoPitch-runs'

def save_to_drive(run_name):
    src = f'/content/runs/{run_name}'
    dst = f'{RUNS_DIR}/{run_name}'
    os.makedirs(RUNS_DIR, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Saved {run_name} → Drive")

def restore_from_drive(run_name):
    src = f'{RUNS_DIR}/{run_name}'
    dst = f'/content/runs/{run_name}'
    os.makedirs(dst, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Restored {run_name} ← Drive")

print("save_to_drive / restore_from_drive ready.")

---
## W1 — Stage 2 setup: restore Stage 1 backbone from Drive

`stage1_conformer_128` was trained locally and uploaded to Drive.
The cell below restores it to `/content/runs/` so Stage 2 can resume from it.

In [ ]:
# stage1_conformer_128:  VDR=89.3%, RPA=99.4%, Med¢=1.3  ← winner
# stage1_conformer_96:   VDR=88.3%, RPA=99.1%, Med¢=2.3

W1_BASE = "stage1_conformer_128"
restore_from_drive(W1_BASE)
print(f"Ready: /content/runs/{W1_BASE}/checkpoints/")

---
## W1 — Stage 2A: joint difflr (technique head, VocalSet + AnnotatedVocalSet)

Differential LR scaled 2× for batch=128 (linear scaling rule: base was bb=5e-5/hd=3e-4 at batch=64).
- `--lr-backbone 1e-4` — 2× base; protects VDR while allowing backbone adaptation
- `--lr 6e-4` — 2× base for heads

Run 1 (bb=5e-5): VDR=91.9% but F1=0.596 — backbone too frozen, technique head underfits.
Run 2 (bb=2e-4): F1=0.741 but VDR drops to ~75% — backbone LR too high.
2× linear scale (bb=1e-4) is the correct middle ground.

In [ ]:
!python vocalcoach/train.py \
    --arch conformer --causal false \
    --hidden 128 --n-layers 4 --n-heads 8 \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset /content/data/annotated_vocalset \
    --output-dir /content/runs/stage2_w1_joint_difflr \
    --epochs 80 --batch-size 128 --num-workers 12 --seq-len 600 \
    --lr 6e-4 --lr-backbone 1e-4 \
    --w-vad 0.05 --w-pitch 2 --w-technique 1 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --metric-vdr-weight 1.0 \
    --patience 25 \
    --resume /content/runs/{W1_BASE}/checkpoints/best_loss.pth

In [ ]:
save_to_drive("stage2_w1_joint_difflr")

---
## W1 — Stage 2B: difflr_v2 (continue from 2A best_metric)

Lower backbone LR (1e-5) to protect VDR if it degraded in 2A.
Resume from `stage2_w1_joint_difflr/best_metric.pth`.

In [ ]:
!python vocalcoach/train.py \
    --arch conformer --causal false \
    --hidden 128 --n-layers 4 --n-heads 8 \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset /content/data/annotated_vocalset \
    --output-dir /content/runs/stage2_w1_difflr_v2 \
    --epochs 60 --batch-size 128 --num-workers 12 --seq-len 600 \
    --lr 2e-4 --lr-backbone 2e-5 \
    --w-vad 0.05 --w-pitch 2 --w-technique 1 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --metric-vdr-weight 1.0 \
    --patience 25 \
    --resume /content/runs/stage2_w1_joint_difflr/checkpoints/best_metric.pth

In [ ]:
save_to_drive("stage2_w1_difflr_v2")

---
## W1 — Stage 2C: GTSinger SupCon (T4 track)

Adds GTSinger technique data + supervised contrastive loss on backbone embeddings.
Resume from best of 2A/2B — check leaderboard first and set `STAGE2_BEST` below.
`--w-contrastive-technique 0.5` is the SupCon loss weight.

In [ ]:
STAGE2_BEST = "stage2_w1_difflr_v2"   # or "stage2_w1_joint_difflr"
restore_from_drive(STAGE2_BEST)

!python vocalcoach/train.py \
    --arch conformer --causal false \
    --hidden 128 --n-layers 4 --n-heads 8 \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset /content/data/annotated_vocalset \
                     /content/data/gtsinger_technique \
    --output-dir /content/runs/stage2_w1_supcon \
    --epochs 50 --batch-size 128 --num-workers 12 --seq-len 600 \
    --lr 2e-4 --lr-backbone 1e-4 \
    --w-vad 0.05 --w-pitch 2 --w-technique 0.5 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --contrastive-technique --w-contrastive-technique 0.5 --contrastive-temp 0.07 \
    --metric-vdr-weight 1.0 \
    --patience 20 \
    --resume /content/runs/{STAGE2_BEST}/checkpoints/best_metric.pth

In [ ]:
save_to_drive("stage2_w1_supcon")

---
## W1 — Stage 2D: quality scoring head (Q1 track)

Adds `--quality-variant` head trained on CCMusicNet + MSE quality data.
Resume from best of 2A/2B (same `STAGE2_BEST` as above — do NOT resume from SupCon run,
quality should branch from the clean technique checkpoint).

Requires `quality_ccmusic.npz` and `quality_mse.npz` — these are **not** in the Drive zip yet.
Add them to the zip and re-upload, or upload separately to Drive before running this cell.

```bash
# Add to zip (local):
zip -1 NanoPitch_data.zip \
    data/quality/quality_ccmusic.npz \
    data/quality/quality_mse.npz
```

In [ ]:
import os
for f in ['quality_ccmusic.npz', 'quality_mse.npz']:
    src = f'{DRIVE_ROOT}/{f}'
    dst = f'/content/data/quality/{f}'
    if not os.path.exists(dst) and os.path.exists(src):
        os.makedirs('/content/data/quality', exist_ok=True)
        import shutil; shutil.copy(src, dst)
        print(f"Copied {f}")

!python vocalcoach/train.py \
    --arch conformer --causal false \
    --hidden 128 --n-layers 4 --n-heads 8 \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset /content/data/annotated_vocalset \
    --quality-dirs /content/data/quality \
    --output-dir /content/runs/stage2_w1_quality \
    --epochs 50 --batch-size 128 --num-workers 12 --seq-len 600 \
    --lr 1e-4 --lr-backbone 5e-5 \
    --w-vad 0.05 --w-pitch 2 --w-technique 0.5 --w-quality 1.0 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --metric-vdr-weight 1.0 \
    --patience 20 \
    --resume /content/runs/{STAGE2_BEST}/checkpoints/best_metric.pth

In [ ]:
save_to_drive("stage2_w1_quality")

---
## Download checkpoint for local evaluation

`update_results.py` must run locally — it writes to your local `VOCALCOACH_RESULTS.md`.
Steps:
1. Run the cell below to copy the checkpoint to Drive
2. Download from Drive to local machine
3. Run `update_results.py` locally with the downloaded checkpoint

**Local eval commands** (run from `~/NanoPitch-MusicalAI`):
```bash
# VocalSet technique eval
python vocalcoach/update_results.py \
    --run-dir vocalcoach/runs/<run_name> \
    --data-dir data \
    --technique-dir data/vocalset \
    --checkpoint best_metric.pth \
    --onset-penalty 1.0 \
    --name <row_label>

# GTSinger technique eval
python vocalcoach/update_results.py \
    --run-dir vocalcoach/runs/<run_name> \
    --data-dir data \
    --gtsinger-technique-dir data/gtsinger_technique \
    --checkpoint best_metric.pth \
    --onset-penalty 1.0 \
    --name <row_label>
```

In [ ]:
import shutil, os
from google.colab import files

# Change RUN_NAME to the run you want to download
RUN_NAME = "stage2_w1_joint_difflr"

# First save to Drive (persistent across sessions)
save_to_drive(RUN_NAME)

# Then also make a flat zip of just the checkpoints for direct download
ckpt_src = f'/content/runs/{RUN_NAME}/checkpoints'
zip_path  = f'/content/{RUN_NAME}_checkpoints.zip'
shutil.make_archive(f'/content/{RUN_NAME}_checkpoints', 'zip', ckpt_src)
print(f"Zipped: {zip_path}")

# Download to browser (triggers file download dialog)
files.download(zip_path)